# Quantitative Trading: Optimal Execution via Almgren-Chriss

## Implementing the Almgren-Chriss Framework for Optimal Trading Execution

This notebook implements the **Almgren-Chriss optimal execution model**, a fundamental framework in quantitative finance that solves the problem of executing a large trade over a specified time horizon while minimizing market impact costs.

### Key Concepts
- **Hamilton-Jacobi-Bellman (HJB) Equation**: Dynamic programming in continuous time
- **Market Impact**: Both temporary (immediate) and permanent (price shift) components
- **Trade-off**: Execute quickly (high impact) vs. execute slowly (higher price drift)

### References
- Almgren, R., & Chriss, N. (2000). "Optimal Execution of Portfolio Transactions"
- Gatheral, J. (2010). "No-dynamic-arbitrage and market impact"


## Part 1: Theory & Mathematical Formulation

### 1.1 The Almgren-Chriss Framework

We want to sell $Q$ shares over time interval $[0, T]$. Define:

- $x(t)$ = cumulative shares sold by time $t$
- $v(t) = dx/dt$ = trading rate (shares/time)
- $S(t)$ = market price at time $t$

**Price Impact Model**:

1. **Permanent Impact** (shift in equilibrium price):
   $$S(t) = S_0 + G(v(t))$$
   where typically $G(v) = \gamma v$ (linear) or $G(v) = \gamma v^2$ (nonlinear)

2. **Temporary Impact** (execution cost above market price):
   $$\text{ExecutionPrice}(t) = S(t) + \lambda v(t)^\alpha$$
   where $\alpha \in [1.5, 2]$ (typically 3/2)

### 1.2 Objective Function

Minimize the **expected execution cost** plus a penalty for **inventory risk**:

$$J = E\left[\int_0^T C(v(t), t) dt + \lambda_{inv} (Q - x(T))^2\right]$$

Where the instantaneous cost is:
$$C(v, t) = \lambda v^\alpha S(t) + \gamma v S(t)^2 \cdot (\text{drift})$$

Rearranging and normalizing, the **discrete-time formulation**:

$$J = \sum_{i=0}^{n-1} \left[ \lambda |v_i|^{1.5} + \gamma v_i \sigma \right] + \eta (Q - \sum_i v_i)^2$$

- $\lambda$ = temporary impact coefficient
- $\gamma$ = permanent impact coefficient  
- $\eta$ = inventory penalty
- $\sigma$ = price volatility


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds
import pandas as pd
from typing import Tuple, Dict

# Set random seed for reproducibility
np.random.seed(42)

## Part 2: Almgren-Chriss Implementation

### 2.1 Model Parameters & Setup

In [ ]:
class AlmgrenChrissExecutor:
    """
    Implements the Almgren-Chriss optimal execution model.
    
    Parameters:
    -----------
    Q : float
        Total quantity to execute (shares)
    S0 : float
        Initial market price
    T : float
        Time horizon (days)
    sigma : float
        Price volatility (annual)
    gamma : float
        Permanent impact coefficient
    lambda_coeff : float
        Temporary impact coefficient
    eta : float
        Inventory penalty coefficient
    n_intervals : int
        Number of time intervals
    """
    
    def __init__(self, Q: float, S0: float, T: float, sigma: float, 
                 gamma: float, lambda_coeff: float, eta: float, n_intervals: int = 50):
        self.Q = Q
        self.S0 = S0
        self.T = T
        self.sigma = sigma  # Annual
        self.gamma = gamma
        self.lambda_coeff = lambda_coeff
        self.eta = eta
        self.n = n_intervals
        self.dt = T / n_intervals
        
    def objective_function(self, v: np.ndarray) -> float:
        """
        Objective: minimize execution cost + inventory risk.
        
        v : array of trading rates (shares/day) for each interval
        """
        # Execution costs
        temp_impact = self.lambda_coeff * np.abs(v) ** 1.5  # Temporary impact
        perm_impact = self.gamma * v  # Permanent impact (price shift)
        
        # Total cost per interval
        costs = temp_impact + perm_impact
        
        # Inventory at end (not fully executed)
        final_inventory = self.Q - np.sum(v)
        inventory_penalty = self.eta * (final_inventory ** 2)
        
        return np.sum(costs) + inventory_penalty
    
    def optimize(self) -> Dict:
        """
        Solve the optimal execution problem using SLSQP.
        """
        # Initial guess: uniform execution
        v0 = np.ones(self.n) * (self.Q / self.n)
        
        # Constraint: sum of execution = Q
        constraints = {'type': 'eq', 'fun': lambda v: np.sum(v) - self.Q}
        
        # Bounds: non-negative execution rates
        bounds = [(0, None) for _ in range(self.n)]
        
        # Optimize
        result = minimize(self.objective_function, v0, method='SLSQP',
                         constraints=constraints, bounds=bounds)
        
        v_optimal = result.x
        
        # Compute trajectories
        x_cumulative = np.cumsum(v_optimal)  # Cumulative executed
        times = np.linspace(0, self.T, self.n)
        
        # Price trajectory with permanent impact
        prices = self.S0 + self.gamma * v_optimal * np.arange(1, self.n + 1)
        
        return {
            'v_optimal': v_optimal,
            'x_cumulative': x_cumulative,
            'times': times,
            'prices': prices,
            'cost': result.fun,
            'success': result.success
        }

# Initialize parameters (realistic market setup)
Q = 1_000_000  # 1M shares
S0 = 100.0     # $100 initial price
T = 1.0        # 1 day execution
sigma = 0.20   # 20% annual volatility
gamma = 1e-6   # Permanent impact
lambda_coeff = 1e-5  # Temporary impact
eta = 1e-6     # Inventory penalty

executor = AlmgrenChrissExecutor(Q, S0, T, sigma, gamma, lambda_coeff, eta, n_intervals=50)
result = executor.optimize()

print(f"Optimal Execution Complete")
print(f"Total Cost: ${result['cost']:.2f}")
print(f"Optimization Success: {result['success']}")
print(f"Total Shares Executed: {result['x_cumulative'][-1]:,.0f}")

### 2.2 Visualization: Optimal Trading Schedule

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Trading Rate (Velocity)
ax = axes[0, 0]
ax.bar(result['times'][:-1], result['v_optimal'], width=executor.dt, alpha=0.7, color='steelblue')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Trading Rate (shares/day)')
ax.set_title('Optimal Trading Schedule: v(t)')
ax.grid(True, alpha=0.3)

# Plot 2: Cumulative Execution
ax = axes[0, 1]
ax.plot(result['times'], result['x_cumulative'], 'o-', color='darkgreen', linewidth=2)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Cumulative Shares Executed')
ax.set_title('Execution Trajectory: x(t)')
ax.grid(True, alpha=0.3)
ax.axhline(y=Q, color='red', linestyle='--', label='Total Target')
ax.legend()

# Plot 3: Price Impact
ax = axes[1, 0]
price_impact = result['prices'] - S0
ax.plot(result['times'], price_impact * 100, 'o-', color='darkred', linewidth=2)  # in bps
ax.set_xlabel('Time (days)')
ax.set_ylabel('Cumulative Price Impact (cents)')
ax.set_title('Price Drift Due to Permanent Impact')
ax.grid(True, alpha=0.3)

# Plot 4: Instantaneous Costs
ax = axes[1, 1]
temp_cost = lambda_coeff * np.abs(result['v_optimal']) ** 1.5
perm_cost = gamma * result['v_optimal']
ax.bar(result['times'][:-1], temp_cost, width=executor.dt, label='Temporary Impact', alpha=0.7, color='orange')
ax.bar(result['times'][:-1], perm_cost, width=executor.dt, bottom=temp_cost, label='Permanent Impact', alpha=0.7, color='purple')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Cost per Interval')
ax.set_title('Decomposition of Execution Costs')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('almgren_chriss_optimal_execution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("OPTIMAL EXECUTION SUMMARY")
print("="*60)

## Part 3: Sensitivity Analysis

### 3.1 Impact Parameter Sensitivity

In [ ]:
# Vary permanent impact parameter
gamma_range = np.logspace(-7, -5, 20)
costs_gamma = []

for g in gamma_range:
    exec_temp = AlmgrenChrissExecutor(Q, S0, T, sigma, g, lambda_coeff, eta, n_intervals=50)
    res = exec_temp.optimize()
    costs_gamma.append(res['cost'])

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(gamma_range, costs_gamma, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Permanent Impact Coefficient (γ)', fontsize=12)
ax.set_ylabel('Total Execution Cost ($)', fontsize=12)
ax.set_title('Sensitivity: Impact of Permanent Impact Parameter', fontsize=14)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('sensitivity_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Cost range: ${min(costs_gamma):.2f} - ${max(costs_gamma):.2f}")
print(f"Cost increases {max(costs_gamma)/min(costs_gamma):.1f}x over γ range")

## Part 4: Comparison with Naive Strategies

### 4.1 Benchmark Strategies

In [ ]:
def uniform_execution(Q, n):
    """Execute uniformly over time."""
    return np.ones(n) * (Q / n)

def front_loaded_execution(Q, n):
    """Execute more aggressively early (assumes adverse drift)."""
    w = np.array([(i+1) / n for i in range(n)])
    return w / np.sum(w) * Q

def back_loaded_execution(Q, n):
    """Execute more aggressively late (assumes positive drift)."""
    w = np.array([1 - (i / n) for i in range(n)])
    return w / np.sum(w) * Q

# Calculate costs for different strategies
v_uniform = uniform_execution(Q, executor.n)
v_front = front_loaded_execution(Q, executor.n)
v_back = back_loaded_execution(Q, executor.n)
v_optimal = result['v_optimal']

cost_uniform = executor.objective_function(v_uniform)
cost_front = executor.objective_function(v_front)
cost_back = executor.objective_function(v_back)
cost_optimal = executor.objective_function(v_optimal)

strategies = ['Uniform', 'Front-Loaded', 'Back-Loaded', 'Almgren-Chriss (Optimal)']
costs = [cost_uniform, cost_front, cost_back, cost_optimal]
improvements = [(c - cost_optimal) / c * 100 for c in costs]

# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cost comparison
colors = ['red', 'orange', 'yellow', 'green']
ax1.bar(strategies, costs, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Total Execution Cost ($)', fontsize=12)
ax1.set_title('Strategy Cost Comparison', fontsize=14)
ax1.grid(True, alpha=0.3, axis='y')
for i, (cost, label) in enumerate(zip(costs, strategies)):
    ax1.text(i, cost, f'${cost:.0f}', ha='center', va='bottom', fontsize=10)

# Improvement relative to optimal
ax2.bar(strategies, [0, improvements[1], improvements[2], improvements[3]], color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Cost Savings vs Strategy (%)', fontsize=12)
ax2.set_title('Optimal vs. Naive Strategies', fontsize=14)
ax2.grid(True, alpha=0.3, axis='y')
for i, (imp, label) in enumerate(zip(improvements, strategies)):
    if imp > 0:
        ax2.text(i, imp, f'{imp:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSTRATEGY COMPARISON")
print("="*70)
for strategy, cost, saving in zip(strategies, costs, improvements):
    print(f"{strategy:25s} | Cost: ${cost:10.2f} | Savings vs. this: {saving:6.2f}%")

## Part 5: Extensions & Real-World Considerations

### 5.1 Stochastic Price Dynamics

In reality, prices follow a stochastic process. The **optimal execution strategy should be adaptive** to realized price changes.

**Geometric Brownian Motion**:
$$dS = \mu S dt + \sigma S dW_t$$

The Almgren-Chriss framework can be extended using **dynamic programming** (HJB equation) to handle stochastic prices. The value function $V(x, S, t)$ (remaining inventory, price, time) satisfies:

$$\frac{\partial V}{\partial t} + \min_v \left[ C(v) + \frac{\partial V}{\partial x} v + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} \right] = 0$$

This is computationally intensive but provides the true optimal adaptive policy.

In [ ]:
# Simulate adaptive execution under price uncertainty
np.random.seed(42)

def simulate_adaptive_execution(n_sims=100):
    """
    Simulate execution under stochastic price dynamics.
    Uses the deterministic optimal policy (simplified).
    """
    dt = executor.dt
    times = np.linspace(0, executor.T, executor.n)
    
    all_prices = []
    all_costs = []
    
    for sim in range(n_sims):
        # Simulate GBM
        dW = np.random.normal(0, np.sqrt(dt), executor.n)
        log_returns = (executor.sigma / np.sqrt(252)) * dW  # Daily volatility
        prices = executor.S0 * np.exp(np.cumsum(log_returns))
        
        # Execute using optimal schedule (not adaptive for simplicity)
        # In reality, would re-optimize at each step
        execution_price = prices + executor.lambda_coeff * np.abs(result['v_optimal']) ** 1.5
        total_cost = np.sum(execution_price * result['v_optimal'])
        
        all_prices.append(prices)
        all_costs.append(total_cost)
    
    return np.array(all_prices), np.array(all_costs), times

prices_sim, costs_sim, times = simulate_adaptive_execution(n_sims=500)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Price paths
for i in range(min(50, len(prices_sim))):
    ax1.plot(times, prices_sim[i], alpha=0.1, color='blue')
ax1.plot(times, np.mean(prices_sim, axis=0), color='red', linewidth=3, label='Mean Price')
ax1.fill_between(times, 
                 np.percentile(prices_sim, 5, axis=0),
                 np.percentile(prices_sim, 95, axis=0),
                 alpha=0.3, color='red', label='90% Confidence Band')
ax1.set_xlabel('Time (days)')
ax1.set_ylabel('Price ($)')
ax1.set_title('Simulated Price Paths (500 scenarios)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cost distribution
ax2.hist(costs_sim, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax2.axvline(np.mean(costs_sim), color='red', linestyle='--', linewidth=2, label=f'Mean: ${np.mean(costs_sim):.2f}')
ax2.axvline(np.median(costs_sim), color='green', linestyle='--', linewidth=2, label=f'Median: ${np.median(costs_sim):.2f}')
ax2.set_xlabel('Total Execution Cost ($)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Execution Costs (500 scenarios)')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('stochastic_execution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSTOCHASTIC EXECUTION ANALYSIS")
print("="*70)
print(f"Mean Execution Cost: ${np.mean(costs_sim):.2f}")
print(f"Std Dev:            ${np.std(costs_sim):.2f}")
print(f"5th Percentile:     ${np.percentile(costs_sim, 5):.2f}")
print(f"95th Percentile:    ${np.percentile(costs_sim, 95):.2f}")
print(f"Max Cost:           ${np.max(costs_sim):.2f}")
print(f"Min Cost:           ${np.min(costs_sim):.2f}")

## Conclusions

1. **Optimal execution is NOT uniform**: The Almgren-Chriss framework shows that naive uniform execution leaves significant costs on the table.

2. **Trade-off between market impact and price risk**: Executing too fast incurs high temporary impact; executing too slow exposes us to adverse price movement.

3. **Nonlinear impact matters**: The temporary impact term scales as $v^{1.5}$ or $v^2$, making very aggressive execution extremely expensive.

4. **Adaptivity is key**: Under stochastic prices, the optimal policy must continuously re-optimize based on observed prices.

5. **Extensions**: Real implementations use HJB dynamic programming, machine learning (reinforcement learning), or Bayesian optimization for true online adaptation.

### Key Takeaway
> **"The cost of execution is not just the bid-ask spread—it's the full impact of your trading on the market. Optimal execution maximizes the value extracted (or minimizes cost paid) by understanding this impact structure."**
